# Fine-tune EfficientNet-B0 for Facial Emotion Recognition

This notebook trains a **7-class emotion classifier** on the **FER+ dataset** (Microsoft's relabeled FER-2013 with crowd-sourced soft labels).

**What you get at the end:**
- `emotion_efficientnet_b0.pth` — trained weights
- `emotion_labels.json` — class-label mapping

Drop both files into `backend/models/` and select *Fine-tuned* in the web UI.

---

**Runtime:** Go to *Runtime → Change runtime type → T4 GPU* before running.

In [ ]:
!pip install -q timm torch torchvision scikit-learn matplotlib pandas

## 1. Load `fer2013.csv` from Google Drive

Mount Drive and copy from your saved location. If the file isn't on Drive, fall back to manual upload.

In [ ]:
import os
import shutil

DRIVE_PATH = "/content/drive/MyDrive/ML_DATAS/Cognitive Science/fer2013.csv"

if not os.path.exists("fer2013.csv"):
    from google.colab import drive
    drive.mount("/content/drive")

    if os.path.exists(DRIVE_PATH):
        shutil.copy(DRIVE_PATH, "fer2013.csv")
        print(f"Copied from Drive: {DRIVE_PATH}")
    else:
        print(f"Not found on Drive at: {DRIVE_PATH}")
        print("Falling back to manual upload:")
        from google.colab import files
        uploaded = files.upload()
        assert "fer2013.csv" in uploaded, "Expected fer2013.csv"
else:
    print("fer2013.csv already present.")

## 2. Download FER+ labels from Microsoft

In [ ]:
import urllib.request

FERPLUS_URL = "https://raw.githubusercontent.com/microsoft/FERPlus/master/fer2013new.csv"
if not os.path.exists("fer2013new.csv"):
    urllib.request.urlretrieve(FERPLUS_URL, "fer2013new.csv")
    print("Downloaded fer2013new.csv")
else:
    print("fer2013new.csv already present.")

## 3. Parse and merge the datasets

In [ ]:
import numpy as np
import pandas as pd

df_pixels = pd.read_csv("fer2013.csv")
df_labels = pd.read_csv("fer2013new.csv")

print(f"fer2013 rows:    {len(df_pixels)}")
print(f"fer2013new rows: {len(df_labels)}")

EMOTION_COLS = ["neutral", "happiness", "surprise", "sadness", "anger", "disgust", "fear"]
LABEL_NAMES  = ["neutral", "happy",     "surprise", "sad",     "angry", "disgust", "fear"]

votes = df_labels[EMOTION_COLS].values.astype(np.float32)
vote_sums = votes.sum(axis=1)

valid_mask = vote_sums > 0
hard_labels = votes[valid_mask].argmax(axis=1)

pixels_series = df_pixels.loc[valid_mask, "pixels"]
usage_series  = df_pixels.loc[valid_mask, "Usage"]

print(f"\nValid samples (with votes): {valid_mask.sum()}")
print(f"Classes: {LABEL_NAMES}")
print(f"Label distribution:\n{pd.Series(hard_labels).value_counts().sort_index()}")

## 4. Build PyTorch Dataset and DataLoaders

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

class FERPlusDataset(Dataset):
    def __init__(self, pixel_strings, labels, transform=None):
        self.pixels = pixel_strings.reset_index(drop=True)
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        px = np.fromstring(self.pixels[idx], sep=" ", dtype=np.uint8).reshape(48, 48)
        img = Image.fromarray(px, mode="L").convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, int(self.labels[idx])


train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

train_mask = usage_series == "Training"
val_mask   = usage_series == "PublicTest"
test_mask  = usage_series == "PrivateTest"

train_ds = FERPlusDataset(pixels_series[train_mask.values], hard_labels[train_mask.values], train_transform)
val_ds   = FERPlusDataset(pixels_series[val_mask.values],   hard_labels[val_mask.values],   val_transform)
test_ds  = FERPlusDataset(pixels_series[test_mask.values],  hard_labels[test_mask.values],  val_transform)

BATCH_SIZE = 64
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"Train: {len(train_ds)}  |  Val: {len(val_ds)}  |  Test: {len(test_ds)}")

## 5. Create the model

In [ ]:
import timm

NUM_CLASSES = len(LABEL_NAMES)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

model = timm.create_model("efficientnet_b0", pretrained=True, num_classes=NUM_CLASSES)
model = model.to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}")

## 6. Training utilities

In [ ]:
from torch import nn, optim
from tqdm.auto import tqdm

criterion = nn.CrossEntropyLoss()

def train_one_epoch(model, loader, optimizer, scheduler=None):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for imgs, labels in tqdm(loader, leave=False, desc="Train"):
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = model(imgs)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        if scheduler:
            scheduler.step()
        total_loss += loss.item() * imgs.size(0)
        correct += (logits.argmax(1) == labels).sum().item()
        total += imgs.size(0)
    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    for imgs, labels in tqdm(loader, leave=False, desc="Eval"):
        imgs, labels = imgs.to(device), labels.to(device)
        logits = model(imgs)
        loss = criterion(logits, labels)
        total_loss += loss.item() * imgs.size(0)
        correct += (logits.argmax(1) == labels).sum().item()
        total += imgs.size(0)
    return total_loss / total, correct / total

## 7. Phase 1 — Train head only (backbone frozen)

In [ ]:
for param in model.parameters():
    param.requires_grad = False
for param in model.classifier.parameters():
    param.requires_grad = True

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Phase 1 trainable params: {trainable:,}")

optimizer = optim.Adam(model.classifier.parameters(), lr=1e-3)
history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

PHASE1_EPOCHS = 5
for epoch in range(PHASE1_EPOCHS):
    t_loss, t_acc = train_one_epoch(model, train_loader, optimizer)
    v_loss, v_acc = evaluate(model, val_loader)
    history["train_loss"].append(t_loss)
    history["train_acc"].append(t_acc)
    history["val_loss"].append(v_loss)
    history["val_acc"].append(v_acc)
    print(f"[Phase 1] Epoch {epoch+1}/{PHASE1_EPOCHS}  "
          f"train_loss={t_loss:.4f}  train_acc={t_acc:.4f}  "
          f"val_loss={v_loss:.4f}  val_acc={v_acc:.4f}")

## 8. Phase 2 — Fine-tune entire network

In [ ]:
for param in model.parameters():
    param.requires_grad = True

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Phase 2 trainable params: {trainable:,}")

optimizer = optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=10 * len(train_loader),
)

best_val_acc = max(history["val_acc"]) if history["val_acc"] else 0.0

PHASE2_EPOCHS = 10
for epoch in range(PHASE2_EPOCHS):
    t_loss, t_acc = train_one_epoch(model, train_loader, optimizer, scheduler)
    v_loss, v_acc = evaluate(model, val_loader)
    history["train_loss"].append(t_loss)
    history["train_acc"].append(t_acc)
    history["val_loss"].append(v_loss)
    history["val_acc"].append(v_acc)
    tag = ""
    if v_acc > best_val_acc:
        best_val_acc = v_acc
        torch.save(model.state_dict(), "emotion_efficientnet_b0.pth")
        tag = "  << saved best"
    print(f"[Phase 2] Epoch {epoch+1}/{PHASE2_EPOCHS}  "
          f"train_loss={t_loss:.4f}  train_acc={t_acc:.4f}  "
          f"val_loss={v_loss:.4f}  val_acc={v_acc:.4f}{tag}")

print(f"\nBest validation accuracy: {best_val_acc:.4f}")

## 9. Training curves

In [ ]:
import matplotlib.pyplot as plt

epochs_range = range(1, len(history["train_loss"]) + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(epochs_range, history["train_loss"], label="Train")
ax1.plot(epochs_range, history["val_loss"],   label="Val")
ax1.axvline(x=PHASE1_EPOCHS + 0.5, color="gray", ls="--", alpha=0.5, label="Phase boundary")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.set_title("Loss")
ax1.legend()

ax2.plot(epochs_range, history["train_acc"], label="Train")
ax2.plot(epochs_range, history["val_acc"],   label="Val")
ax2.axvline(x=PHASE1_EPOCHS + 0.5, color="gray", ls="--", alpha=0.5, label="Phase boundary")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Accuracy")
ax2.set_title("Accuracy")
ax2.legend()

plt.tight_layout()
plt.show()

## 10. Evaluate on test set + confusion matrix

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

model.load_state_dict(torch.load("emotion_efficientnet_b0.pth", map_location=device, weights_only=True))
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for imgs, labels in tqdm(test_loader, desc="Test"):
        imgs = imgs.to(device)
        preds = model(imgs).argmax(1).cpu()
        all_preds.append(preds)
        all_labels.append(labels)

all_preds  = torch.cat(all_preds).numpy()
all_labels = torch.cat(all_labels).numpy()

test_acc = (all_preds == all_labels).mean()
print(f"Test accuracy: {test_acc:.4f}\n")
print(classification_report(all_labels, all_preds, target_names=LABEL_NAMES))

cm = confusion_matrix(all_labels, all_preds)
fig, ax = plt.subplots(figsize=(8, 8))
ConfusionMatrixDisplay(cm, display_labels=LABEL_NAMES).plot(ax=ax, cmap="Blues", colorbar=False)
ax.set_title("Confusion Matrix (Test Set)")
plt.tight_layout()
plt.show()

## 11. Export model and labels

In [ ]:
import json

json.dump(LABEL_NAMES, open("emotion_labels.json", "w"))
print("Saved: emotion_labels.json")
print("Saved: emotion_efficientnet_b0.pth")
print(f"\nLabel mapping: {dict(enumerate(LABEL_NAMES))}")
print("\nDownload both files and place them in backend/models/")

In [ ]:
files.download("emotion_efficientnet_b0.pth")
files.download("emotion_labels.json")